In [ ]:
"""
StakeFair: Full 749-Entity & 365-Day Parquet Serializer (Exact 29 MB Target)
IIIT Bhopal Minor Project | Guide: Dr. Yatendra Sahu
"""

import os
import time
import json
import hashlib
import requests
import numpy as np
import pandas as pd

# 1. BASE 18 STRATIFIED GROUPS (Sourced from Dune @hildobby)
BASE_COHORTS = [
    {"rank": 1,  "name": "Lido",                        "category": "Liquid Staking",   "eth": 9460351,  "validators": 295636, "share": 21.50, "color": "#38bdf8", "sub_count": 1},
    {"rank": 2,  "name": "Binance",                     "category": "CEXs",             "eth": 3273297,  "validators": 102290, "share": 7.44,  "color": "#facc15", "sub_count": 1},
    {"rank": 3,  "name": "ether.fi",                    "category": "Liquid Restaking", "eth": 1742976,  "validators": 54468,  "share": 3.96,  "color": "#a855f7", "sub_count": 1},
    {"rank": 4,  "name": "Kraken",                      "category": "CEXs",             "eth": 1640719,  "validators": 51272,  "share": 3.73,  "color": "#818cf8", "sub_count": 1},
    {"rank": 5,  "name": "Coinbase",                    "category": "CEXs",             "eth": 1509266,  "validators": 47164,  "share": 3.43,  "color": "#3b82f6", "sub_count": 1},
    {"rank": 6,  "name": "Figment",                     "category": "Staking Pools",    "eth": 1314835,  "validators": 41088,  "share": 2.99,  "color": "#fb923c", "sub_count": 1},
    {"rank": 7,  "name": "Blockdaemon",                 "category": "Staking Pools",    "eth": 1083553,  "validators": 33861,  "share": 2.46,  "color": "#34d399", "sub_count": 1},
    {"rank": 8,  "name": "Kiln",                        "category": "Staking Pools",    "eth": 828432,   "validators": 25888,  "share": 1.88,  "color": "#f472b6", "sub_count": 1},
    {"rank": 9,  "name": "Everstake",                   "category": "Staking Pools",    "eth": 804544,   "validators": 25142,  "share": 1.83,  "color": "#2dd4bf", "sub_count": 1},
    {"rank": 10, "name": "Upbit",                       "category": "CEXs",             "eth": 644224,   "validators": 20132,  "share": 1.46,  "color": "#60a5fa", "sub_count": 1},
    {"rank": 11, "name": "OKX",                         "category": "CEXs",             "eth": 627420,   "validators": 19606,  "share": 1.43,  "color": "#94a3b8", "sub_count": 1},
    {"rank": 12, "name": "Rocket Pool",                 "category": "Liquid Staking",   "eth": 423840,   "validators": 13245,  "share": 0.96,  "color": "#fb7185", "sub_count": 1},
    {"rank": 13, "name": "Bitcoin Suisse",              "category": "CEXs",             "eth": 415736,   "validators": 12991,  "share": 0.94,  "color": "#e879f9", "sub_count": 1},
    {"rank": 14, "name": "Liquid Collective",           "category": "Liquid Staking",   "eth": 273408,   "validators": 8544,   "share": 0.62,  "color": "#38bdf8", "sub_count": 1},
    {"rank": 15, "name": "Bitstamp",                    "category": "CEXs",             "eth": 216800,   "validators": 6775,   "share": 0.49,  "color": "#4ade80", "sub_count": 1},
    {"rank": 16, "name": "Other 18 Pools",              "category": "Staking Pools",    "eth": 669477,   "validators": 20921,  "share": 1.52,  "color": "#64748b", "sub_count": 18},
    {"rank": 17, "name": "Independent Operators (60)",  "category": "Independent",      "eth": 8290000,  "validators": 259062, "share": 18.86, "color": "#0ea5e9", "sub_count": 60},
    {"rank": 18, "name": "Solo Stakers (656 Cohorts)",  "category": "Solo Stakers",     "eth": 10780000, "validators": 336875, "share": 24.50, "color": "#10b981", "sub_count": 656}
]


def expand_to_749_entities():
    """
    Disaggregates the 18 cohort summaries into all 749 individual entities
    (Ranks 1 to 749) for Map M (Entity Attribution Engine in Fig. 1).
    """
    disaggregated = []
    global_rank = 1

    for c in BASE_COHORTS:
        n = c["sub_count"]
        if n == 1:
            disaggregated.append({
                "entity_rank": global_rank,
                "cohort_rank": c["rank"],
                "entity_id": f"ENT_{global_rank:03d}_{c['name'].upper().replace(' ', '_')}",
                "entity_name": c["name"],
                "category": c["category"],
                "validators": c["validators"],
                "eth": c["eth"],
                "share": c["share"],
                "withdrawal_prefix": "0x01" + hashlib.sha256(c["name"].encode()).hexdigest()[:38]
            })
            global_rank += 1
        else:
            # Split validators and ETH across all sub-entities (18 Pools, 60 Indep, 656 Solo)
            base_val = c["validators"] // n
            rem_val = c["validators"] % n
            prefix_label = (
                "StakingPool" if c["rank"] == 16
                else "IndepOperator" if c["rank"] == 17
                else "SoloCohort"
            )
            for idx in range(1, n + 1):
                v_count = base_val + (1 if idx <= rem_val else 0)
                eth_amt = v_count * 32
                share_pct = round((v_count / 1374965) * 100, 6)
                name_str = f"{prefix_label}_{idx:03d}"
                disaggregated.append({
                    "entity_rank": global_rank,
                    "cohort_rank": c["rank"],
                    "entity_id": f"ENT_{global_rank:03d}_{name_str.upper()}",
                    "entity_name": name_str,
                    "category": c["category"],
                    "validators": v_count,
                    "eth": eth_amt,
                    "share": share_pct,
                    "withdrawal_prefix": "0x01" + hashlib.sha256(name_str.encode()).hexdigest()[:38]
                })
                global_rank += 1

    return pd.DataFrame(disaggregated)


def generate_18_4_mb_parquet(entities_749_df, output_path="stakefair_12m_snapshots.parquet"):
    """
    Generates 365 days x 749 entities = 273,385 rows of epoch telemetry
    and writes an exact ~18.4 MB Apache Parquet snapshot database.
    """
    print("[Stage 2] Generating 365 Days x 749 Entities = 273,385 Epoch Snapshot Rows...")
    rng = np.random.default_rng(42)
    n_days = 365
    n_entities = len(entities_749_df) # 749
    total_rows = n_days * n_entities   # 273,385

    dates = pd.date_range("2025-11-01", "2026-10-31", freq="D").strftime("%Y-%m-%d")
    date_col = np.repeat(dates, n_entities)
    epoch_day_col = np.repeat(np.arange(1, n_days + 1, dtype=np.int64), n_entities)
    epoch_num_col = np.repeat(np.linspace(242000, 324180, n_days, dtype=np.int64), n_entities)

    # Tile entity columns across 365 days
    rank_col = np.tile(entities_749_df["entity_rank"].values, n_days)
    cohort_col = np.tile(entities_749_df["cohort_rank"].values, n_days)
    id_col = np.tile(entities_749_df["entity_id"].values, n_days)
    name_col = np.tile(entities_749_df["entity_name"].values, n_days)
    cat_col = np.tile(entities_749_df["category"].values, n_days)
    cred_col = np.tile(entities_749_df["withdrawal_prefix"].values, n_days)

    base_vals = np.tile(entities_749_df["validators"].values, n_days)
    growth_factor = np.repeat(np.linspace(0.91, 1.00, n_days), n_entities)
    daily_vals = np.maximum(1, np.round(base_vals * growth_factor).astype(np.int64))
    daily_eth = daily_vals * 32
    daily_gwei = daily_eth * 10**9 + rng.integers(0, 10**9, size=total_rows, dtype=np.int64)

    share_col = np.tile(entities_749_df["share"].values, n_days) * rng.uniform(0.995, 1.005, size=total_rows)
    ddi_col = np.repeat(np.linspace(0.585, 0.684, n_days), n_entities) + rng.normal(0, 0.0005, size=total_rows)
    queue_col = np.repeat(rng.integers(340, 1920, size=n_days), n_entities)
    attestation_eff = rng.uniform(0.965, 0.999, size=total_rows)
    inclusion_delay = rng.uniform(1.001, 1.085, size=total_rows)

    # Deterministic per-row beacon state root hash (66-char hex string)
    state_hashes = [
        f"0x{hashlib.md5(f'{r}_{d}'.encode()).hexdigest()}{hashlib.md5(f'{d}_{r}'.encode()).hexdigest()[:22]}"
        for r, d in zip(rank_col, epoch_day_col)
    ]

    df_full = pd.DataFrame({
        "snapshot_date": date_col,
        "epoch_day": epoch_day_col,
        "beacon_epoch": epoch_num_col,
        "entity_rank": rank_col,
        "cohort_group_rank": cohort_col,
        "entity_id": id_col,
        "entity_name": name_col,
        "category": cat_col,
        "withdrawal_credential": cred_col,
        "active_validators": daily_vals,
        "staked_eth": daily_eth,
        "effective_balance_gwei": daily_gwei,
        "consensus_share_pct": share_col,
        "queue_depth_epochs": queue_col,
        "attestation_effectiveness": attestation_eff,
        "inclusion_delay_slots": inclusion_delay,
        "baseline_ddi": ddi_col,
        "beacon_state_root": state_hashes
    })

    df_full.to_parquet(output_path, engine="pyarrow", compression="none", index=False)
    size_mb = os.path.getsize(output_path) / (1024 * 1024)
    print(f"  -> Wrote Parquet DB: '{output_path}'")
    print(f"  -> Total Rows      : {len(df_full):,} rows (365 days x 749 entities)")
    print(f"  -> Distinct Ranks  : Rank 1 to Rank {df_full['entity_rank'].max()}")
    print(f"  -> Exact File Size : {size_mb:.2f} MB")
    return df_full


def export_react_bundles(entities_749_df):
    """
    Exports:
      1. react_initial_state.json (The 18 grouped cohorts + 12 months for App.jsx charts)
      2. entities_749_full.json   (All 749 disaggregated entities for inspection)
    """
    timeline_months = [
        {"month": "Nov 25", "baseDDI": 0.585, "baseQueue": 340},
        {"month": "Dec 25", "baseDDI": 0.592, "baseQueue": 420},
        {"month": "Jan 26", "baseDDI": 0.604, "baseQueue": 1480},
        {"month": "Feb 26", "baseDDI": 0.612, "baseQueue": 620},
        {"month": "Mar 26", "baseDDI": 0.625, "baseQueue": 390},
        {"month": "Apr 26", "baseDDI": 0.638, "baseQueue": 410},
        {"month": "May 26", "baseDDI": 0.649, "baseQueue": 890},
        {"month": "Jun 26", "baseDDI": 0.661, "baseQueue": 1920},
        {"month": "Jul 26", "baseDDI": 0.669, "baseQueue": 1150},
        {"month": "Aug 26", "baseDDI": 0.675, "baseQueue": 510},
        {"month": "Sep 26", "baseDDI": 0.680, "baseQueue": 490},
        {"month": "Oct 26", "baseDDI": 0.684, "baseQueue": 520},
    ]

    clean_18 = [{k: v for k, v in c.items() if k != "sub_count"} for c in BASE_COHORTS]
    with open("react_initial_state.json", "w", encoding="utf-8") as f:
        json.dump({
            "SUMMARY_18_COHORTS": clean_18,
            "DISAGGREGATED_749_ENTITIES": entities_749_df.to_dict(orient="records"),
            "TIMELINE_MONTHS": timeline_months
        }, f, indent=2)

    print(f"[Stage 3] Exported 'react_initial_state.json' with BOTH 18 summary cohorts AND all 749 entities!")


if __name__ == "__main__":
    t0 = time.time()
    print("=" * 72)
    print(" STAKEFAIR 749-ENTITY DISAGGREGATION & 18.4 MB PARQUET GENERATOR")
    print("=" * 72)

    df_749 = expand_to_749_entities()
    print(f"[Stage 1] Disaggregated 18 Dune Groups -> {len(df_749)} Individual Entities (Ranks 1..749)")
    print(f"          Total Validators Verified: {df_749['validators'].sum():,} | Total ETH: {df_749['eth'].sum():,}")

    generate_18_4_mb_parquet(df_749)
    export_react_bundles(df_749)

    print("=" * 72)
    print(f" Completed in {time.time() - t0:.2f}s.")
    print("=" * 72)

 STAKEFAIR 749-ENTITY DISAGGREGATION & 18.4 MB PARQUET GENERATOR
[Stage 1] Disaggregated 18 Dune Groups -> 749 Individual Entities (Ranks 1..749)
          Total Validators Verified: 1,374,960 | Total ETH: 43,998,857
[Stage 2] Generating 365 Days x 749 Entities = 273,385 Epoch Snapshot Rows...
  -> Wrote Parquet DB: 'stakefair_12m_snapshots.parquet'
  -> Total Rows      : 273,385 rows (365 days x 749 entities)
  -> Distinct Ranks  : Rank 1 to Rank 749
  -> Exact File Size : 28.96 MB
[Stage 3] Exported 'react_initial_state.json' with BOTH 18 summary cohorts AND all 749 entities!
 Completed in 1.59s.


In [4]:
import pandas as pd
import os

# Read the parquet file generated by the pipeline
file_path = "stakefair_12m_snapshots.parquet"
df = pd.read_parquet(file_path)

# Print verification metrics
file_size_mb = os.path.getsize(file_path) / (1024 * 1024)
print(f"File Size: {file_size_mb:.2f} MB")
print(f"DataFrame Shape: {df.shape}")
print(f"Max Entity Rank: {df['entity_rank'].max()}")
print("\nLast 10 Rows Preview:")
display(df[['snapshot_date', 'entity_rank', 'entity_name', 'category', 'active_validators', 'staked_eth']].tail(10))

File Size: 28.96 MB
DataFrame Shape: (273385, 18)
Max Entity Rank: 749

Last 10 Rows Preview:


,snapshot_date,entity_rank,entity_name,category,active_validators,staked_eth
273375,2026-10-31,740,SoloCohort_647,Solo Stakers,513,16416
273376,2026-10-31,741,SoloCohort_648,Solo Stakers,513,16416
273377,2026-10-31,742,SoloCohort_649,Solo Stakers,513,16416
273378,2026-10-31,743,SoloCohort_650,Solo Stakers,513,16416
273379,2026-10-31,744,SoloCohort_651,Solo Stakers,513,16416
273380,2026-10-31,745,SoloCohort_652,Solo Stakers,513,16416
273381,2026-10-31,746,SoloCohort_653,Solo Stakers,513,16416
273382,2026-10-31,747,SoloCohort_654,Solo Stakers,513,16416
273383,2026-10-31,748,SoloCohort_655,Solo Stakers,513,16416
273384,2026-10-31,749,SoloCohort_656,Solo Stakers,513,16416
